# Mini Project 1 — Star dataset analysis

**Dataset:** [Kaggle Star Dataset](https://www.kaggle.com/datasets/deepu1109/star-dataset?resource=download) (local file: `stars.csv` in this folder)

**Sections:** 1. Overview · 2. Data Profile · 3. Analysis · 4. Conclusions · 5. Process

Open and run this notebook from the `MP1/` directory so paths resolve correctly.

In [ ]:
# Setup (required - do not delete)
!pip install -q jupyter plotly kaleido pandas

## Section 1 — Overview

This notebook analyzes a table of 240 stars with temperature, luminosity, radius, absolute magnitude, color label, spectral class, and a numeric star-type code (0–5: brown dwarf through hypergiant).

I chose this dataset because I am interested in astronomy and in practice turning scientific measurements into visuals that are easier to reason about. The three questions below guide the analysis:

1. **Which star properties are most useful for telling star types apart?** Compare temperature, luminosity, radius, and absolute magnitude across types.
2. **Are larger stars always brighter, or does the relationship change by star type?** Look at radius vs luminosity overall and by type.
3. **Does a star's color reliably indicate its temperature and type?** Compare cleaned color labels with temperature and type, and note overlap.

These questions matter for HCD-style work because they connect what users *see* (color, relative brightness) to underlying measurements—and they show where a single cue is not enough.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

CHARTS_DIR = Path("charts")
CHARTS_DIR.mkdir(exist_ok=True)
DATA_PATH = Path("stars.csv")
EXPORT_WIDTH = 1200
EXPORT_HEIGHT = 700

STAR_TYPE_LABELS = {
    0: "Brown Dwarf",
    1: "Red Dwarf",
    2: "White Dwarf",
    3: "Main Sequence",
    4: "Supergiant",
    5: "Hypergiant",
}

STAR_TYPE_ORDER = [
    "Brown Dwarf",
    "Red Dwarf",
    "White Dwarf",
    "Main Sequence",
    "Supergiant",
    "Hypergiant",
]

COLOR_NORMALIZATION = {
    "blue white": "Blue-White",
    "blue-white": "Blue-White",
    "bluewhite": "Blue-White",
    "blue": "Blue",
    "red": "Red",
    "white": "White",
    "white-yellow": "White-Yellow",
    "yellow-white": "Yellow-White",
    "yellowish white": "Yellow-White",
    "yellowish": "Yellowish",
    "yellowish-orange": "Yellowish-Orange",
    "orange": "Orange",
    "orange-red": "Orange-Red",
    "pale yellow orange": "Pale Yellow-Orange",
    "whitish": "Whitish",
}


def clean_star_color(value: str) -> str:
    key = str(value).strip().lower().replace("  ", " ").replace("_", " ")
    return COLOR_NORMALIZATION.get(key, str(value).strip().title())


df = pd.read_csv(DATA_PATH)
df["Star_type_label"] = df["Star type"].map(STAR_TYPE_LABELS)
df["Star_color_clean"] = df["Star color"].apply(clean_star_color)
df["log10_Luminosity"] = df["Luminosity(L/Lo)"].where(df["Luminosity(L/Lo)"] > 0).apply(
    lambda v: np.log10(v) if pd.notna(v) else np.nan
)
df["log10_Radius"] = df["Radius(R/Ro)"].where(df["Radius(R/Ro)"] > 0).apply(
    lambda v: np.log10(v) if pd.notna(v) else np.nan
)

## Section 2 — Data Profile

Run four standard checks before analysis. Each result gets one sentence of interpretation below.

In [ ]:
df.head()

**`head()`:** The table has one row per star with mixed numeric and text columns; the first rows look like real measurements, not placeholders.

In [ ]:
df.info()

**`info()`:** There are 240 rows and 7 original columns (plus derived label columns); dtypes are mostly float64 for measurements and object for color/spectral class.

In [ ]:
df.describe()

**`describe()`:** Numeric columns span huge ranges (especially luminosity and radius), which suggests log scales or standardization will be needed in charts.

In [ ]:
df.isnull().sum()

**`isnull().sum()`:** There are no missing values in the core columns, so group means and correlations are not distorted by gaps in this file.

## Section 3 — Analysis

Three charts—one per question. Each is saved to `charts/` with Kaleido and shown below as a static image for GitHub readers.

### Question 1 — Which star properties best separate star types?

**Chart type:** Heatmap of z-scored mean values by star type.

**What this chart argues:** Properties with strong color contrast across rows do more work separating types.

In [ ]:
metric_cols = [
    "Temperature (K)",
    "log10_Luminosity",
    "log10_Radius",
    "Absolute magnitude(Mv)",
]

means = (
    df.groupby("Star_type_label", observed=True)[metric_cols]
    .mean()
    .reindex(STAR_TYPE_ORDER)
)
zscores = (means - means.mean()) / means.std(ddof=0)
zscores = zscores.rename(
    columns={
        "log10_Luminosity": "Luminosity (L/Lo, log10)",
        "log10_Radius": "Radius (R/Ro, log10)",
        "Absolute magnitude(Mv)": "Absolute Magnitude (Mv)",
    }
)

fig_q1 = go.Figure(
    data=[
        go.Heatmap(
            z=zscores.values,
            x=zscores.columns.tolist(),
            y=zscores.index.tolist(),
            zmid=0,
            colorscale="RdBu",
            reversescale=True,
            text=np.round(zscores.values, 2),
            texttemplate="%{text}",
            colorbar={"title": "Z-score"},
        )
    ]
)
fig_q1.update_layout(
    title="Q1: Star-Type Separation Profile Across Key Properties",
    xaxis_title="Star Property",
    yaxis_title="Star Type",
    width=EXPORT_WIDTH,
    height=EXPORT_HEIGHT,
)
q1_path = CHARTS_DIR / "q1_type_separation.png"
fig_q1.write_image(q1_path, scale=2)
fig_q1.show()
print("Saved:", q1_path.resolve())

![Q1: type separation heatmap](charts/q1_type_separation.png)

**Interpretation:** Luminosity and radius show the clearest contrast between dwarf classes and giant/hypergiant classes. Temperature helps but is not sufficient alone. Star type is best read using several properties together.

### Question 2 — Are larger stars always brighter?

**Chart type:** Log-log scatter (radius vs luminosity, colored by star type).

In [ ]:
r_raw = df["Radius(R/Ro)"].corr(df["Luminosity(L/Lo)"])
r_log = df["log10_Radius"].corr(df["log10_Luminosity"])
print(f"Pearson r (raw): {r_raw:.3f}")
print(f"Pearson r (log10): {r_log:.3f}")

fig_q2 = px.scatter(
    df,
    x="Radius(R/Ro)",
    y="Luminosity(L/Lo)",
    color="Star_type_label",
    log_x=True,
    log_y=True,
    category_orders={"Star_type_label": STAR_TYPE_ORDER},
    hover_data=["Temperature (K)", "Absolute magnitude(Mv)", "Star_color_clean", "Spectral Class"],
    title="Q2: Radius vs Luminosity by Star Type (Log-Log Scale)",
    labels={
        "Radius(R/Ro)": "Radius (R/Ro, log scale)",
        "Luminosity(L/Lo)": "Luminosity (L/Lo, log scale)",
        "Star_type_label": "Star Type",
    },
)
fig_q2.update_layout(width=EXPORT_WIDTH, height=EXPORT_HEIGHT)
q2_path = CHARTS_DIR / "q2_radius_luminosity.png"
fig_q2.write_image(q2_path, scale=2)
fig_q2.show()
print("Saved:", q2_path.resolve())

![Q2: radius vs luminosity](charts/q2_radius_luminosity.png)

**Interpretation:** There is a strong upward trend on the log-log plot (correlation about 0.92 on log values vs about 0.53 on raw values). Larger stars are often brighter here, but points cluster by star type, so one simple rule does not fit all types.

### Question 3 — Does color reliably indicate temperature and type?

**Chart type:** Box plot of temperature by cleaned color label.

In [ ]:
color_order = (
    df.groupby("Star_color_clean", observed=True)["Temperature (K)"]
    .mean()
    .sort_values()
    .index.tolist()
)

fig_q3 = px.box(
    df,
    x="Star_color_clean",
    y="Temperature (K)",
    color="Star_type_label",
    category_orders={"Star_color_clean": color_order, "Star_type_label": STAR_TYPE_ORDER},
    title="Q3: Temperature Distribution by Star Color and Star Type",
    labels={
        "Star_color_clean": "Star Color (cleaned)",
        "Temperature (K)": "Temperature (K)",
        "Star_type_label": "Star Type",
    },
)
fig_q3.update_layout(width=EXPORT_WIDTH, height=EXPORT_HEIGHT, xaxis_tickangle=-35)
q3_path = CHARTS_DIR / "q3_color_temperature.png"
fig_q3.write_image(q3_path, scale=2)
fig_q3.show()
print("Saved:", q3_path.resolve())

display(
    df.groupby("Star_color_clean", observed=True)["Temperature (K)"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean")
    .round(1)
)

![Q3: temperature by color](charts/q3_color_temperature.png)

**Interpretation:** Red-labeled stars are cooler on average; blue and blue-white groups are hotter. Boxes still overlap, so color points in the right direction but should be combined with other measurements, not used alone to assign type.

## Section 4 — Conclusions

### Question 1
Luminosity and radius are the strongest separators of star type in this dataset, especially on log scales. Temperature and absolute magnitude add context but do not tell the full story by themselves. If I had more time, I would test a simple classifier using combinations of these columns.

### Question 2
Radius and luminosity are strongly related overall on a log-log view, so larger stars are often brighter—but the relationship is not uniform across types. I would next model by star type to quantify where the rule breaks down.

### Question 3
Color aligns with temperature in the expected direction after cleaning messy label spellings, but overlap between groups means color is a weak standalone predictor of type. Further work could merge color with spectral class or use continuous temperature instead of text color alone.

## Section 5 — Process

**Week 4–5:** I picked the Kaggle star dataset and locked three analytical questions. I used pandas first: head, info, missing counts, value_counts, a hot/bright filter, and groupby means by star type.

**Week 6 (A6):** I moved to Plotly and exported static PNGs with Kaleido. The first Q1 chart was a faceted box plot; the exported PNG had overlapping labels, so I switched to a z-score heatmap. I cleaned color strings and used log10 for luminosity and radius.

**Tools:** Cursor helped with boilerplate and export wiring; I chose chart types, read the outputs, and decided when the Q1 design was not good enough to submit.

**This folder:** stars.csv and charts/ are duplicated here so MP1 runs standalone. Older matplotlib plots live in archive/legacy/ and are not part of the submission.